In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from textwrap import fill
from torch import norm


#LOAD AUXILIARY DATA
#Load the OAC data
oa_lsoa = pd.read_csv('../data/geofiles/lookup_oa2022_lsoa11_EW.csv')
oa_lsoa.set_index('OA21CD',inplace=True)
oa_msoa = pd.read_csv('../data/geofiles/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv')[["OA21CD","MSOA21CD"]].set_index('OA21CD')

#Load the IMD data
imd = pd.read_csv('../data/geofiles/uk_imd2019.csv')
imd = imd[["LSOA","SOA_decile"]]
imd.columns = ['LSOA11CD','IMD']

#Load the population density data
density = pd.read_csv('../data/census_data/eng_raw_csvs/ts006.csv')
density.columns = ['OA21CD','Density']
density['Density_decile'] = pd.qcut(density['Density'],10,labels=False)
density['Density_decile'] = 10 - density['Density_decile'] #reverse the order
density.drop('Density',axis=1,inplace=True)
density.set_index('OA21CD',inplace=True)


/tmp/ipykernel_1427/676632053.py:14: DtypeWarning: Columns (0: LSOA21NMW, 1: MSOA21NMW, 2: LAD22NMW) have mixed types. Specify dtype option on import or set low_memory=False.
  oa_msoa = pd.read_csv('../data/geofiles/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv')[["OA21CD","MSOA21CD"]].set_index('OA21CD')


In [2]:
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
data = pd.read_parquet(cleaned_data_path)
# Ensure OA is a column, not just an index
data = data.reset_index()

reco_err = pd.DataFrame()
pca_reco_err = pd.DataFrame()
# bottleneck_sizes = [2,4,8,16,32,64,100,128]
bottleneck_sizes = [100]
for bottleneck in bottleneck_sizes:
    print(f"Processing bottleneck: {bottleneck}")
    # Load reconstructed autoencoder data
    reco_path = f"../AE_outputs/engcensus_all/250epoch_scan_lin/census_geodemo__bottleneck_{bottleneck}_v1__reconstructed_outputs.csv"
    reco_data = pd.read_csv(reco_path, index_col=0)
    # Load PCA reconstruction data
    pca_path = f"../AE_outputs/engcensus_all/PCA/{bottleneck}_components.csv"
    pca_reco = pd.read_csv(pca_path, index_col=0)

    # # #compute rmse
    ae_err = np.sqrt(np.mean((data.set_index("OA") - reco_data) ** 2, axis=1)).reset_index()
    pca_err = np.sqrt(np.mean((data.set_index("OA") - pca_reco) ** 2, axis=1)).reset_index()
    # Rename columns
    ae_err = ae_err.rename(columns={0: f"reco_err_{bottleneck}"})
    pca_err = pca_err.rename(columns={0: f"pca_err_{bottleneck}"})

    #multiply the errors by 100 to get percentage errors
    ae_err[f"reco_err_{bottleneck}"] = ae_err[f"reco_err_{bottleneck}"] * 100
    pca_err[f"pca_err_{bottleneck}"] = pca_err[f"pca_err_{bottleneck}"] * 100

    # Normalize errors
    ae_err[f"reco_err_{bottleneck}_norm"] = ae_err[f"reco_err_{bottleneck}"] / ae_err[f"reco_err_{bottleneck}"].mean() * 100
    pca_err[f"pca_err_{bottleneck}_norm"] = pca_err[f"pca_err_{bottleneck}"] / pca_err[f"pca_err_{bottleneck}"].mean() * 100


    #normed diffs againist each other

    ae_err[f"perc_diff_{bottleneck}_ae"] = (ae_err[f"reco_err_{bottleneck}"] - pca_err[f"pca_err_{bottleneck}"])/pca_err[f"pca_err_{bottleneck}"] * 100
    pca_err[f"perc_diff_{bottleneck}_pca"] = (pca_err[f"pca_err_{bottleneck}"] - ae_err[f"reco_err_{bottleneck}"])/ae_err[f"reco_err_{bottleneck}"] * 100
    

    # Merge into final DataFrames
    if reco_err.empty:
        reco_err = ae_err
        pca_reco_err = pca_err
    else:
        reco_err = reco_err.merge(ae_err, on="OA")
        pca_reco_err = pca_reco_err.merge(pca_err, on="OA")

    print(f"Number of OAs where AE is greater than PCA: {len(ae_err[ae_err[f'reco_err_{bottleneck}'] > pca_err[f'pca_err_{bottleneck}']])} out of {len(ae_err)}")
    print("which is", len(ae_err[ae_err[f'reco_err_{bottleneck}'] > pca_err[f'pca_err_{bottleneck}']]) / len(ae_err) * 100, "% of the OAs")
    #print the PCAs

#rename OA to OA21CD
reco_err.rename(columns={"OA": "OA21CD"}, inplace=True)
pca_reco_err.rename(columns={"OA": "OA21CD"}, inplace=True)

Processing bottleneck: 100
Number of OAs where AE is greater than PCA: 13062 out of 188880
which is 6.915501905972047 % of the OAs


In [4]:
pca_err["pca_err_100"].mean()

np.float64(0.8939997629959616)

In [5]:
# Load OAC data
OAC = pd.read_csv("../data/OAC/OAC_assignment.csv")
OAC = OAC[["Geography_Code", "Supergroup8", "Group", "Subgroup"]]
OAC = OAC.rename(columns={"Geography_Code": "OA21CD"})
# Load the data/OAC_cats.csv
OAC_cats = pd.read_csv("../data/OAC/OAC_cats.csv")
OAC_cats = OAC_cats[['Classification Code', 'Classification Name']]

# Make a dict out of the first 8 rows
OAC_cats_dict = OAC_cats.set_index('Classification Code')['Classification Name'].to_dict()
# Convert codes to string before mapping
OAC['Supergroup8'] = OAC['Supergroup8'].astype(str)
OAC['Supergroup_name'] = OAC['Supergroup8'].map(OAC_cats_dict)
#create a column which is Code + Name
OAC['Supergroup_codename'] = OAC['Supergroup8'] + " - " + OAC['Supergroup_name']
OAC['Group'] = OAC['Group'].astype(str)
OAC['Group_name'] = OAC['Group'].map(OAC_cats_dict)
OAC['Subgroup'] = OAC['Subgroup'].astype(str)
OAC['Subgroup_name'] = OAC['Subgroup'].map(OAC_cats_dict)

# Merge OAC data
reco_err = reco_err.merge(OAC, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(OAC, on="OA21CD", how="left")

# Add density deciles
reco_err = reco_err.merge(density, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(density, on="OA21CD", how="left")
# Add LSOA
reco_err = reco_err.merge(oa_lsoa, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(oa_lsoa, on="OA21CD", how="left")
# Add MSOA
reco_err = reco_err.merge(oa_msoa, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(oa_msoa, on="OA21CD", how="left")
reco_err = reco_err.dropna()
# Add IMD
reco_err = reco_err.merge(imd, on="LSOA11CD", how="left").dropna()
pca_reco_err = pca_reco_err.merge(imd, on="LSOA11CD", how="left").dropna()
reco_err["IMD"] = reco_err["IMD"].astype(int)
pca_reco_err["IMD"] = pca_reco_err["IMD"].astype(int)

#check len is 188880
if len(reco_err) != 188880 or len(pca_reco_err) != 188880:
    print(f"Length of reco_err is {len(reco_err)}")
    print(f"Length of pca_reco_err is {len(pca_reco_err)}")
    raise ValueError("Dataframes do not match expected length of 188880")


# Plotting Scripts

In [ ]:


def prepare_grouped_data(df1, df2, group_col, bottleneck, norm=False):
    """Merge two datasets and compute mean reconstruction error for each group."""

    if norm:
        ae_col = f"reco_err_{bottleneck}_norm"
        pca_col = f"pca_err_{bottleneck}_norm"
    else:
        ae_col = f"reco_err_{bottleneck}"
        pca_col = f"pca_err_{bottleneck}"
    
    grouped_1 = df1.groupby(group_col)[ae_col].mean().reset_index()
    grouped_2 = df2.groupby(group_col)[pca_col].mean().reset_index()
    
    grouped = grouped_1.merge(grouped_2, on=group_col)
    grouped.rename(columns={ae_col: "AE", pca_col: "PCA"}, inplace=True)
    
    grouped_melted = grouped.melt(id_vars=[group_col], var_name="Error Type", value_name="Mean Reco Error")
    return grouped, grouped_melted


def plot_combined_chart(df1, df2, group_col, xlabel, title_prefix, palette, bottleneck, wrapped_labels=False):
    grouped, grouped_melted = prepare_grouped_data(df1, df2, group_col, bottleneck)
    grouped["Error Difference"] = grouped["AE"] - grouped["PCA"]
    grouped["Perc_Error_Change"] = (grouped["AE"] - grouped["PCA"]) / grouped["PCA"] * 100
    fig, axes = plt.subplots(2, 1, figsize=(8, 7), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

    # Top: Side-by-side bar chart
    sns.barplot(ax=axes[0], x=group_col, y="Mean Reco Error", hue="Error Type", data=grouped_melted, palette=palette)
    axes[0].set_xlabel("")
    axes[0].set_ylabel("% of Mean Reconstruction Error")
    axes[0].set_title(f"{title_prefix} (Bottleneck {bottleneck})")
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(title="Error Type", frameon=False)

    # Bottom: Error difference
    if norm:
        sns.barplot(ax=axes[1], x=group_col, y="Error Difference", data=grouped, color="darkblue")
    else:
        sns.barplot(ax=axes[1], x=group_col, y="Perc_Error_Change", data=grouped, color="darkblue")
    axes[1].axhline(grouped["Perc_Error_Change"].mean(), color="red", linestyle="--")
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel("Difference (AE - PCA) [%]")
    axes[1].tick_params(axis='x', rotation=30)

    if wrapped_labels:
        axes[0].set_xticklabels([fill(label, width=10) for label in grouped[group_col]])
        axes[1].set_xticklabels([fill(label, width=10) for label in grouped[group_col]])

    plt.tight_layout()
    plt.savefig(f"../plots/comparison_{xlabel}_bottleneck_{bottleneck}.png", dpi=300)
    plt.show()

def plot_side_by_side_charts(df1, df2, group_col1, group_col2, xlabel1, xlabel2, title_prefix, palette, bottleneck, norm=False, rotation=45, wrapped_labels=False, print_means=False):
    """Creates a figure with two side-by-side subplots for IMD and Density Decile, with optional mean value printing."""
    
    grouped1, grouped_melted1 = prepare_grouped_data(df1, df2, group_col1, bottleneck, norm=norm)
    grouped2, grouped_melted2 = prepare_grouped_data(df1, df2, group_col2, bottleneck, norm=norm)
    grouped1["Error Difference"] = grouped1["AE"] - grouped1["PCA"]
    grouped1["Perc_Error_Change"] = (grouped1["AE"] - grouped1["PCA"]) / grouped1["PCA"] * 100
    grouped2["Error Difference"] = grouped2["AE"] - grouped2["PCA"]
    grouped2["Perc_Error_Change"] = (grouped2["AE"] - grouped2["PCA"]) / grouped2["PCA"] * 100

    if print_means:
        def print_summary(grouped_melted, group_col, label):
            means = grouped_melted.groupby([group_col, "Error Type"])["Mean Reco Error"].mean().unstack()
            print(f"\n=== Mean Reconstruction Error by Group ({label}) ===")
            print(means)

            if grouped_melted[group_col].dtype.name == "category":
                levels = grouped_melted[group_col].cat.categories
            else:
                levels = sorted(grouped_melted[group_col].unique())

            if len(levels) >= 2:
                first, last = levels[0], levels[-1]
                print(f"\n--- Percentage Increase in Error ({label}): {last} → {first} ---")
                perc_diffs = []
                for error_type in means.columns:
                    first_val = means.loc[first, error_type]
                    last_val = means.loc[last, error_type]
                    perc_diff = ((first_val - last_val) / last_val) * 100
                    print(f"{error_type}: {perc_diff:.2f}%")#
                    perc_diffs.append((error_type, perc_diff))
                print("\n--- Summary ---")
                for error_type, perc_diff in perc_diffs:
                    if perc_diff > 0:
                        print(f"{error_type} shows a {perc_diff:.2f}% increase in error from {last} to {first}.")
                    elif perc_diff < 0:
                        print(f"{error_type} shows a {abs(perc_diff):.2f}% decrease in error from {last} to {first}.")
                    else:
                        print(f"{error_type} shows no change in error from {last} to {first}.")
                

             


        print_summary(grouped_melted1, group_col1, xlabel1)
        print_summary(grouped_melted2, group_col2, xlabel2)

    
    fig, axes = plt.subplots(2, 2, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]}, sharex='col')
    
    sns.barplot(ax=axes[0, 0], x=group_col1, y="Mean Reco Error", hue="Error Type", data=grouped_melted1, palette=palette)
    axes[0, 0].set_xlabel("")
    axes[0, 0].set_ylabel("% of Mean Reconstruction Error" if norm else "Mean Reconstruction Error (RMSE)")
    axes[0, 0].set_title(f"{title_prefix} by {xlabel1}")
    axes[0, 0].tick_params(axis='x', rotation=rotation)
    axes[0, 0].legend(title="Error Type", frameon=False)
    
    sns.barplot(ax=axes[0, 1], x=group_col2, y="Mean Reco Error", hue="Error Type", data=grouped_melted2, palette=palette)
    axes[0, 1].set_xlabel("")
    axes[0, 1].set_ylabel("")
    axes[0, 1].set_title(f"{title_prefix} by {xlabel2}")
    axes[0, 1].tick_params(axis='x', rotation=rotation)
    axes[0, 1].legend(title="Error Type", frameon=False)

    if norm:
        sns.barplot(ax=axes[1, 0], x=group_col1, y="Error Difference", data=grouped1, color="darkblue")
        axes[1, 0].axhline(0, color="red", linestyle="--")
        axes[1, 0].set_ylabel("Difference (AE - PCA) [%]")
    else:
        sns.barplot(ax=axes[1, 0], x=group_col1, y="Perc_Error_Change", data=grouped1, color="darkblue")
        mean_val1 = grouped1["Perc_Error_Change"].mean()
        axes[1, 0].axhline(mean_val1, color="red", linestyle="--", label=f'Mean: {mean_val1:.1f}%')
        axes[1, 0].set_ylabel("(AE - PCA)/PCA [%]")
        axes[1, 0].legend(frameon=False, fontsize=8)
    axes[1, 0].set_xlabel(xlabel1)
    axes[1, 0].tick_params(axis='x', rotation=rotation)

    if norm:
        sns.barplot(ax=axes[1, 1], x=group_col2, y="Error Difference", data=grouped2, color="darkblue")
        axes[1, 1].axhline(0, color="red", linestyle="--")
        axes[1, 1].set_ylabel("Difference (AE - PCA) [%]")
    else:
        sns.barplot(ax=axes[1, 1], x=group_col2, y="Perc_Error_Change", data=grouped2, color="darkblue")
        mean_val2 = grouped2["Perc_Error_Change"].mean()
        axes[1, 1].axhline(mean_val2, color="red", linestyle="--", label=f'Mean: {mean_val2:.1f}%')
        axes[1, 1].set_ylabel("(AE - PCA)/PCA [%]")
        axes[1, 1].legend(frameon=False, fontsize=8)
    axes[1, 1].set_xlabel(xlabel2)
    axes[1, 1].tick_params(axis='x', rotation=0)

    if wrapped_labels:
        axes[0, 0].set_xticklabels([fill(label, width=21) for label in grouped1[group_col1]])
        axes[1, 0].set_xticklabels([fill(label, width=21) for label in grouped1[group_col1]])
        axes[0, 1].set_xticklabels([fill(label, width=21) for label in grouped2[group_col2]])
        axes[1, 1].set_xticklabels([fill(label, width=21) for label in grouped2[group_col2]])
    
    plt.tight_layout()
    plt.savefig(f"../plots/comparison_{xlabel1}_{xlabel2}_bottleneck_{bottleneck}.png", dpi=300)
    plt.show()

# Produce Fig 4 & 5

In [10]:
# Set seaborn style
sns.set(style="white")
#
# Color palette
palette = {
    "AE": "seagreen",
    "PCA": "sandybrown"
}

for bottleneck in bottleneck_sizes:
    print(f"Processing bottleneck: {bottleneck}")
    #print line of hashes and big gap
    print("#" * 50)
    print("#" * 50)
    print("#" * 50)
    plot_side_by_side_charts(
        reco_err, 
        pca_reco_err, 
        "IMD", 
        "Density_decile", 
        "IMD Decile", 
        "Density Decile", 
        "Comparison of Reconstruction Error", 
        palette,
        bottleneck=bottleneck,
        rotation=0,
        print_means=True
    )

    plot_side_by_side_charts(
        reco_err, 
        pca_reco_err, 
        "Supergroup_codename", 
        "Group", 
        "OAC SuperGroup", 
        "OAC group", 
        "Comparison of Reconstruction Error", 
        palette,
        bottleneck=bottleneck, 
        rotation=60,
        wrapped_labels=True
    )
    


Processing bottleneck: 100
##################################################
##################################################
##################################################

=== Mean Reconstruction Error by Group (IMD Decile) ===
Error Type        AE       PCA
IMD                           
1           0.902292  1.021146
2           0.869339  0.988226
3           0.854406  0.970913
4           0.825822  0.938972
5           0.798797  0.899005
6           0.782260  0.878930
7           0.751787  0.836593
8           0.738848  0.823429
9           0.722552  0.801474
10          0.697128  0.774880

--- Percentage Increase in Error (IMD Decile): 10 → 1 ---


AttributeError: 'list' object has no attribute 'columns'

In [8]:
"""
This script compares reconstruction errors between two models—baseline reconstruction ('reco_err')
and PCA-based reconstruction ('pca_reco_err')—at two geographic levels:
  1. Output Area (OA21CD)
  2. Middle Layer Super Output Area (MSOA21CD)

It computes the percentage difference in reconstruction error for each model and outputs spatial
GeoDataFrames for mapping or further spatial analysis.

Steps:
- Merge and align error data
- Attach geographic boundaries (OAs and MSOAs)
- Calculate % difference in error across different bottleneck sizes
- Export the results to Parquet files for mapping
"""

# STEP 1: Merge reconstruction errors at OA level
# Merge reco_err with pca_reco_err on OA21CD, keeping suffixes for conflicting column names
merged = reco_err.merge(pca_reco_err, on="OA21CD", suffixes=("", "_pca"))

# Remove duplicated columns from pca_reco_err unless they're actual PCA error columns
merged = merged[[col for col in merged.columns if not col.endswith("_pca") or col.startswith("pca_err")]]

# STEP 2: Load OA boundaries and merge with error data
oa_gdf = gpd.read_parquet("../data/geofiles/oabounds_2021approx.parquet")[["OA", "geometry"]]
oa_gdf = oa_gdf.rename(columns={"OA": "OA21CD"})  # match OA code field name
merged = oa_gdf.merge(merged, on="OA21CD", how="right")  # keep only OAs with error data

# STEP 3: Calculate % difference in error between reco_err and pca_err for each bottleneck size
for b in bottleneck_sizes:
    merged[f"perc_diff_{b}"] = (
        (merged[f"reco_err_{b}"] - merged[f"pca_err_{b}"]) / merged[f"pca_err_{b}"] * 100
    )

# STEP 4: Format and export OA-level results
merged.set_index("OA21CD", inplace=True)
merged = merged.round(5)
merged.to_parquet("../data/plots/maps/error_diff_by_OA.parquet")

# STEP 5: Aggregate to MSOA level
# Keep only error columns and MSOA codes for aggregation
merged = merged[[col for col in merged.columns if col.startswith("reco_err") or col.startswith("pca_err") or col == "MSOA21CD"]]

# Group by MSOA and compute mean error values
msoa = merged.groupby("MSOA21CD").mean().reset_index()

# Recompute % difference in error at MSOA level
for b in bottleneck_sizes:
    msoa[f"perc_diff_{b}"] = (
        (msoa[f"reco_err_{b}"] - msoa[f"pca_err_{b}"]) / msoa[f"pca_err_{b}"] * 100
    )

# STEP 6: Load MSOA boundaries and merge with error data
msoa_gdf = gpd.read_file(
    "../data/geofiles/Middle_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFE_(V8)_and_RUC"
)[["MSOA21CD", "geometry"]]
msoa = msoa_gdf.merge(msoa, on="MSOA21CD", how="right")

# STEP 7: Export MSOA-level GeoDataFrame
msoa.to_parquet("../data/plots/maps/error_diff_by_MSOA.parquet")


FileNotFoundError: [Errno 2] Failed to open local file '../data/plots/maps/error_diff_by_OA.parquet'. Detail: [errno 2] No such file or directory

In [ ]:
#Calculate quantiles for negative values in MSOA perc_diff_100 to make figure 7


# Filter for negative values only
neg_values = msoa[msoa["perc_diff_100"] < 0]["perc_diff_100"]
print("Only",len(msoa)-len(neg_values),"out of",len(msoa),"MSOAs have higher AE reconstruction error than PCA reconstruction error")
# Calculate quantile breaks (e.g., 4 breaks = quartiles)
quantiles = neg_values.quantile([0, 0.2, 0.4, 0.6, 0.8, 1])
print("Quantiles for negative perc_diff_100 values to make figure 7: \n", quantiles)
